In [1]:
#Imports
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [2]:
#Configure headers
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [3]:
#Define a class
class Website:
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers = headers)
        soup = BeautifulSoup(response.content, "html.parser")
        self.title = soup.title.string if soup.title else "No title found"
        
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        
        self.text = soup.body.get_text(separator = "\n", strip = True)

In [4]:
#Define system prompt for better context setting
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [5]:
#A function that writes a User Prompt that asks for summaries of websites
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [6]:
#A function which constructs message for our summarizer
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [11]:
#Main summarizer function
OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

def summarize(url):
    website = Website(url)
    payload = {"model": MODEL, "messages": messages_for(website), "stream": False}
    response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
    return response.json()['message']['content']

In [12]:
#Function to show summary in markdown style
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [13]:
display_summary("https://edwarddonner.com/")

**Summary**
================

### About the Author

* Ed is a writer, code enthusiast, and AI expert.
* He co-founded Nebula.io, applying AI to help people discover their potential.
* He has also founded an AI startup, untapt, which was acquired in 2021.

### News and Announcements
---------------------------

* **September 15, 2025**: "AI in Production: Gen AI and Agentic AI on AWS at scale" (event announcement)
* **May 28, 2025**: "Connecting my courses – become an LLM expert and leader" (announcement about online courses)
* **May 18, 2025**: "2025 AI Executive Briefing" (announcement about a workshop or briefing)
* **April 21, 2025**: "The Complete Agentic AI Engineering Course" (announcement about an online course)

### Website Features
-------------------

* Connect Four: an arena where LLMs compete against each other.
* Outsmart: an arena that pits LLMs against each other in a battle of diplomacy and deviousness.
* About page with personal information and background on Nebula.io.
* Contact form for getting in touch with Ed.
* Links to social media profiles and newsletter subscription.